In [ ]:
# AI Protein Property Predictor: membrane vs non-membrane (UniProt, human, reviewed)
# In Google Colab, run this whole file as one cell, or paste it block by block.
# Needs: pandas numpy scikit-learn matplotlib  (all already installed in Colab)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score, accuracy_score)

SEED = 42                                  # fixed seed = same results every run
np.random.seed(SEED)
AA = list("ACDEFGHIKLMNPQRSTVWY")          # the 20 amino acids

In [ ]:
# ---------------- STEP 1: download data ----------------
URL = ("https://rest.uniprot.org/uniprotkb/stream?format=tsv"
       "&query=reviewed:true+AND+organism_id:9606"
       "&fields=accession,sequence,length,cc_subcellular_location,xref_pfam")
df = pd.read_csv(URL, sep="\t")
df.to_csv("uniprot_human_raw.csv", index=False)      # saved copy for reproducibility
print("Downloaded proteins:", df.shape[0])
print("Columns:", df.columns.tolist())

In [ ]:
# ---------------- STEP 2: labels (membrane = 1, not = 0) ----------------
loc_col = [c for c in df.columns if "ubcellular" in c][0]   # finds the location column
seq_col = [c for c in df.columns if c.lower() == "sequence"][0]
id_col = df.columns[0]
pfam_col = [c for c in df.columns if "pfam" in c.lower()][0]

df = df.dropna(subset=[loc_col, seq_col]).copy()
df["label"] = df[loc_col].str.contains("membrane", case=False).astype(int)
print("\nClass counts (0 = not membrane, 1 = membrane):")
print(df["label"].value_counts())

In [ ]:
# ---------------- STEP 3: features (amino acid fractions + length) ----------------
def composition(seq):
    seq = "".join(c for c in seq if c in AA)         # drop unusual letters like X, U
    n = max(len(seq), 1)
    return [seq.count(a) / n for a in AA] + [len(seq)]

X = np.array([composition(s) for s in df[seq_col]])
y = df["label"].values
feature_names = AA + ["length"]

In [ ]:
# ---------------- STEP 4: fair split (protein families stay together) ----------------
pfam = df[pfam_col].fillna("").str.split(";").str[0]
groups = np.where(pfam == "", df[id_col], pfam)      # no Pfam -> its own group
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr, X_te, y_tr, y_te = X[train_idx], X[test_idx], y[train_idx], y[test_idx]
print("\nTrain size:", len(y_tr), " Test size:", len(y_te))

In [ ]:
# ---------------- STEP 5: train two models ----------------
lr = make_pipeline(StandardScaler(),
                   LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                            n_jobs=-1, random_state=SEED)
lr.fit(X_tr, y_tr)
rf.fit(X_tr, y_tr)

In [ ]:
# ---------------- STEP 6: evaluate ----------------
results = []
for name, model in [("Logistic Regression", lr), ("Random Forest", rf)]:
    pred = model.predict(X_te)
    print("\n==", name, "==")
    print(classification_report(y_te, pred, target_names=["not membrane", "membrane"]))
    results.append([name, round(accuracy_score(y_te, pred), 3), round(f1_score(y_te, pred), 3)])
    ConfusionMatrixDisplay(confusion_matrix(y_te, pred),
                           display_labels=["not membrane", "membrane"]).plot()
    plt.title(name)
    plt.savefig(name.replace(" ", "_") + "_confusion_matrix.png", dpi=150)
    plt.show()
print(pd.DataFrame(results, columns=["Model", "Accuracy", "F1 (membrane)"]))

In [ ]:
# ---------------- STEP 7: interpret (which letters mattered?) ----------------
imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values()
imp.plot.barh(figsize=(6, 6), title="Random Forest feature importance")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()
print("\nTop 8 important features:")
print(imp.sort_values(ascending=False).head(8))

# Direction of effect from Logistic Regression (+ = pushes towards membrane)
coef = pd.Series(lr[-1].coef_[0], index=feature_names).sort_values()
print("\nMost 'membrane-like' features (LR):", coef.tail(5).index.tolist())
print("Most 'non-membrane-like' features (LR):", coef.head(5).index.tolist())

In [ ]:
# ---------------- BONUS: random split vs family split (data leakage demo) ----------------
from sklearn.model_selection import train_test_split
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
rf2 = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                             n_jobs=-1, random_state=SEED).fit(Xa, ya)
print("\nRF F1 with RANDOM split :", round(f1_score(yb, rf2.predict(Xb)), 3))
print("RF F1 with FAMILY split :", round(f1_score(y_te, rf.predict(X_te)), 3))
print("Random split usually looks better because similar proteins leak into the test set.")